# Platform Adapters Demo

**Sprint**: 3 — Multi-Platform Support  
**Feature**: Platform-Specific Data Normalization  
**Purpose**: Demonstrate the TwitterAdapter and TelegramAdapter classes that normalize raw API data into a common BriefingInput format for the CDDBS analysis pipeline.

---

## Overview

This notebook demonstrates:
1. **Twitter API v2 data normalization** — Tweets, user profiles, engagement metrics
2. **Telegram MTProto data normalization** — Messages, channels, forwarding metadata
3. **Format comparison** — Side-by-side comparison of normalized output
4. **Edge case handling** — Missing fields, malformed data, media types
5. **Integration with quality scorer** — Feed normalized data into the scoring pipeline

In [ ]:
# === Cell 1: Imports and Setup ===
# Import the platform adapters from the tools module

import sys
import json
from pathlib import Path
from datetime import datetime

# Add project root to path so we can import from tools/
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from tools.platform_adapters import (
    TwitterAdapter,
    TelegramAdapter,
    PostData,
    ProfileData,
    BriefingInput,
)

twitter_adapter = TwitterAdapter()
telegram_adapter = TelegramAdapter()

print(f"TwitterAdapter platform:  {twitter_adapter.platform}")
print(f"TelegramAdapter platform: {telegram_adapter.platform}")
print(f"\nData classes available: PostData, ProfileData, BriefingInput")

## 1. Twitter API v2 Data Normalization

Simulates raw data from the Twitter API v2 user timeline endpoint.
The adapter normalizes this into our common `BriefingInput` format.

In [ ]:
# === Cell 2: Sample Twitter API v2 Data ===
# Simulates raw response from Twitter API v2 endpoints
# Modeled on the @rt_com account from our high_quality_briefing fixture

raw_twitter_data = {
    'profile': {
        'id': '64643056',
        'username': 'rt_com',
        'name': 'RT',
        'description': 'RT is the first Russian 24/7 English-language news channel which brings the Russian view on global news.',
        'created_at': '2009-03-14T10:30:00Z',
        'verified': False,
        'is_blue_verified': False,
        'lang': 'en',
        'public_metrics': {
            'followers_count': 3180000,
            'following_count': 1247,
            'tweet_count': 258000,
            'listed_count': 12500,
        },
    },
    'posts': [
        {
            'id': '1893456789012345678',
            'text': 'NATO expansion continues to threaten European security. Our analysis of the latest summit decisions. https://t.co/abc123',
            'created_at': '2026-02-15T12:00:00Z',
            'lang': 'en',
            'public_metrics': {
                'like_count': 1250,
                'retweet_count': 890,
                'reply_count': 234,
                'quote_count': 67,
                'impression_count': 450000,
            },
            'entities': {
                'urls': [{'expanded_url': 'https://rt.com/news/nato-summit-analysis/', 'url': 'https://t.co/abc123'}],
                'mentions': [{'username': 'NATO'}],
            },
            'referenced_tweets': [],
            'attachments': {},
        },
        {
            'id': '1893456789012345679',
            'text': 'RT @SputnikInt: Breaking: EU sanctions backfire as energy costs surge across Europe. Full report: https://t.co/def456',
            'created_at': '2026-02-15T14:30:00Z',
            'lang': 'en',
            'public_metrics': {
                'like_count': 340,
                'retweet_count': 520,
                'reply_count': 89,
                'quote_count': 12,
                'impression_count': 180000,
            },
            'entities': {
                'urls': [{'expanded_url': 'https://sputniknews.com/eu-sanctions/', 'url': 'https://t.co/def456'}],
                'mentions': [{'username': 'SputnikInt'}],
            },
            'referenced_tweets': [{'type': 'retweeted', 'id': '1893456789012345000', 'author_username': 'SputnikInt'}],
            'attachments': {},
        },
        {
            'id': '1893456789012345680',
            'text': 'Western media refuses to show the full picture. Here is what they are not telling you about the conflict.',
            'created_at': '2026-02-15T17:00:00Z',
            'lang': 'en',
            'public_metrics': {
                'like_count': 2100,
                'retweet_count': 1450,
                'reply_count': 567,
                'quote_count': 89,
                'impression_count': 780000,
            },
            'entities': {'urls': [], 'mentions': []},
            'referenced_tweets': [],
            'attachments': {'media_keys': ['3_1893456789012345680']},
        },
    ],
    'collection_period': {'start': '2026-01-01', 'end': '2026-02-15'},
    'data_source': 'twitter_api_v2',
}

print(f"Raw Twitter data loaded:")
print(f"  Profile: @{raw_twitter_data['profile']['username']}")
print(f"  Posts: {len(raw_twitter_data['posts'])}")
print(f"  Collection period: {raw_twitter_data['collection_period']}")

In [ ]:
# === Cell 3: Normalize Twitter Data ===
# Run the TwitterAdapter to produce a BriefingInput

twitter_input = twitter_adapter.normalize(raw_twitter_data)

print("Normalized Twitter BriefingInput")
print("=" * 60)

# Profile
p = twitter_input.profile
print(f"\nProfile:")
print(f"  Handle:     {p.handle}")
print(f"  Platform:   {p.platform}")
print(f"  Name:       {p.display_name}")
print(f"  Bio:        {p.bio[:80]}...")
print(f"  Followers:  {p.followers:,}")
print(f"  Following:  {p.following:,}")
print(f"  Created:    {p.created_at}")
print(f"  Verified:   {p.verified}")
print(f"  Metadata:   {p.platform_metadata}")

# Posts
print(f"\nPosts ({len(twitter_input.posts)}):")
for i, post in enumerate(twitter_input.posts):
    print(f"\n  Post {i+1}:")
    print(f"    ID:           {post.post_id}")
    print(f"    Text:         {post.text[:70]}...")
    print(f"    Timestamp:    {post.timestamp}")
    print(f"    Media type:   {post.media_type}")
    print(f"    Amplification:{post.is_amplification} (source: {post.amplification_source or 'N/A'})")
    print(f"    Engagement:   {post.engagement}")
    print(f"    URLs:         {post.urls}")
    print(f"    Mentions:     {post.mentions}")

print(f"\nData source: {twitter_input.data_source}")
print(f"Collection:  {twitter_input.collection_period}")

## 2. Telegram MTProto Data Normalization

Simulates raw data from Telegram's MTProto API (via Telethon).
Key differences from Twitter:
- Messages have views and forwards instead of likes/retweets
- Forwarded messages retain source channel metadata
- Channels don't "follow" — they have subscribers
- Creation date is not directly exposed

In [ ]:
# === Cell 4: Sample Telegram MTProto Data ===
# Simulates raw response from Telethon/Pyrogram
# Modeled on the @rt_english channel from our telegram fixture

raw_telegram_data = {
    'profile': {
        'id': -1001234567890,
        'username': 'rt_english',
        'title': 'RT English',
        'description': 'RT is a global news network. Watch RT LIVE at rt.com. Follow us for breaking news and commentary.',
        'type': 'channel',
        'members_count': 524000,
        'message_count': 45000,
        'date': 1577836800,  # Unix timestamp (2020-01-01)
        'verified': False,
        'linked_chat_username': 'rt_english_chat',
        'invite_link': 't.me/rt_english',
    },
    'posts': [
        {
            'message_id': 45001,
            'text': 'NATO expansion continues to threaten European security. Our analysis of the latest summit decisions. https://rt.com/news/nato-summit-analysis/',
            'date': '2026-02-15T11:57:00Z',
            'views': 95000,
            'forwards': 234,
            'reply_count': 45,
            'entities': [
                {'type': 'url', 'offset': 95, 'length': 46},
            ],
        },
        {
            'message_id': 45002,
            'text': 'Breaking: EU sanctions backfire as energy costs surge across Europe.',
            'date': '2026-02-15T14:25:00Z',
            'views': 78000,
            'forwards': 189,
            'reply_count': 32,
            'forward_from_chat': {
                'id': -1001234567891,
                'username': 'rt_russian',
                'title': 'RT Russian',
            },
            'forward_date': '2026-02-15T14:20:00Z',
            'entities': [],
        },
        {
            'message_id': 45003,
            'text': 'Watch: Western media refuses to show the full picture about the conflict. @journalist_name provides on-the-ground reporting.',
            'date': '2026-02-15T16:55:00Z',
            'views': 120000,
            'forwards': 567,
            'reply_count': 89,
            'video': {'file_id': 'BAACAgIAAxk...', 'duration': 180},
            'entities': [
                {'type': 'mention', 'offset': 73, 'length': 16},
            ],
        },
    ],
    'collection_period': {'start': '2026-01-15', 'end': '2026-02-15'},
    'data_source': 'telegram_mtproto',
}

print(f"Raw Telegram data loaded:")
print(f"  Channel: @{raw_telegram_data['profile']['username']}")
print(f"  Messages: {len(raw_telegram_data['posts'])}")
print(f"  Subscribers: {raw_telegram_data['profile']['members_count']:,}")

In [ ]:
# === Cell 5: Normalize Telegram Data ===
# Run the TelegramAdapter to produce a BriefingInput

telegram_input = telegram_adapter.normalize(raw_telegram_data)

print("Normalized Telegram BriefingInput")
print("=" * 60)

# Profile
p = telegram_input.profile
print(f"\nProfile:")
print(f"  Handle:     {p.handle}")
print(f"  Platform:   {p.platform}")
print(f"  Name:       {p.display_name}")
print(f"  Bio:        {p.bio[:80]}...")
print(f"  Followers:  {p.followers:,}  (subscribers)")
print(f"  Following:  {p.following}  (channels don't follow)")
print(f"  Created:    {p.created_at}")
print(f"  Verified:   {p.verified}")
print(f"  Metadata:   {json.dumps(p.platform_metadata, indent=4)}")

# Posts
print(f"\nMessages ({len(telegram_input.posts)}):")
for i, post in enumerate(telegram_input.posts):
    print(f"\n  Message {i+1}:")
    print(f"    ID:           {post.post_id}")
    print(f"    Text:         {post.text[:70]}...")
    print(f"    Timestamp:    {post.timestamp}")
    print(f"    Media type:   {post.media_type}")
    print(f"    Forwarded:    {post.is_amplification} (from: {post.amplification_source or 'N/A'})")
    print(f"    Engagement:   {post.engagement}")
    print(f"    URLs:         {post.urls}")
    print(f"    Mentions:     {post.mentions}")

## 3. Side-by-Side Comparison

Compare how the same content appears after normalization from each platform.
This demonstrates the adapter pattern: different raw formats → identical output structure.

In [ ]:
# === Cell 6: Side-by-Side Field Comparison ===
# Compare normalized outputs field by field

def compare_profiles(twitter: ProfileData, telegram: ProfileData):
    """Side-by-side comparison of normalized profiles."""
    fields = [
        ('Handle', twitter.handle, telegram.handle),
        ('Platform', twitter.platform, telegram.platform),
        ('Display Name', twitter.display_name, telegram.display_name),
        ('Bio (first 50)', twitter.bio[:50], telegram.bio[:50]),
        ('Followers/Subs', f"{twitter.followers:,}", f"{telegram.followers:,}"),
        ('Following', str(twitter.following), str(telegram.following)),
        ('Created', twitter.created_at, telegram.created_at),
        ('Verified', str(twitter.verified), str(telegram.verified)),
        ('Language', twitter.language, telegram.language),
    ]
    
    print(f"{'Field':20s} | {'Twitter':35s} | {'Telegram':35s}")
    print("-" * 95)
    for field_name, tw_val, tg_val in fields:
        print(f"{field_name:20s} | {str(tw_val):35s} | {str(tg_val):35s}")


def compare_posts(twitter_post: PostData, telegram_post: PostData):
    """Side-by-side comparison of normalized posts."""
    fields = [
        ('Post ID', twitter_post.post_id, telegram_post.post_id),
        ('Text (first 50)', twitter_post.text[:50], telegram_post.text[:50]),
        ('Timestamp', str(twitter_post.timestamp)[:19], str(telegram_post.timestamp)[:19]),
        ('Media Type', twitter_post.media_type, telegram_post.media_type),
        ('Is Amplification', str(twitter_post.is_amplification), str(telegram_post.is_amplification)),
        ('Amp Source', twitter_post.amplification_source or 'N/A', telegram_post.amplification_source or 'N/A'),
    ]
    
    print(f"{'Field':20s} | {'Twitter':35s} | {'Telegram':35s}")
    print("-" * 95)
    for field_name, tw_val, tg_val in fields:
        print(f"{field_name:20s} | {str(tw_val):35s} | {str(tg_val):35s}")
    
    # Engagement metrics differ by platform
    print(f"\n  Engagement (platform-specific):")
    print(f"    Twitter:  {twitter_post.engagement}")
    print(f"    Telegram: {telegram_post.engagement}")


print("Profile Comparison")
print("=" * 95)
compare_profiles(twitter_input.profile, telegram_input.profile)

print(f"\n\nPost Comparison (Post #1)")
print("=" * 95)
compare_posts(twitter_input.posts[0], telegram_input.posts[0])

print(f"\n\nPost Comparison (Post #2 — Amplification)")
print("=" * 95)
compare_posts(twitter_input.posts[1], telegram_input.posts[1])

## 4. Edge Case Handling

Test that the adapters handle missing or malformed data gracefully.

In [ ]:
# === Cell 7: Edge Case Testing ===
# Test adapters with minimal/missing data

test_cases = [
    {
        'name': 'Empty profile (Twitter)',
        'adapter': twitter_adapter,
        'data': {'profile': {}, 'posts': []},
    },
    {
        'name': 'Empty profile (Telegram)',
        'adapter': telegram_adapter,
        'data': {'profile': {}, 'posts': []},
    },
    {
        'name': 'Missing metrics (Twitter)',
        'adapter': twitter_adapter,
        'data': {
            'profile': {'username': 'test_user', 'name': 'Test'},
            'posts': [{'id': '123', 'text': 'Hello world', 'created_at': '2026-01-01T00:00:00Z'}],
        },
    },
    {
        'name': 'Unix timestamp date (Telegram)',
        'adapter': telegram_adapter,
        'data': {
            'profile': {'username': 'test_channel', 'title': 'Test', 'date': 1704067200},
            'posts': [{'message_id': 1, 'text': 'Test message', 'date': 1704067200}],
        },
    },
    {
        'name': 'Media types (Telegram)',
        'adapter': telegram_adapter,
        'data': {
            'profile': {'username': 'media_channel'},
            'posts': [
                {'message_id': 1, 'text': 'Photo post', 'photo': [{'file_id': 'abc'}]},
                {'message_id': 2, 'text': 'Video post', 'video': {'file_id': 'def'}},
                {'message_id': 3, 'text': 'Document', 'document': {'file_id': 'ghi'}},
                {'message_id': 4, 'text': 'Voice message', 'voice': {'file_id': 'jkl'}},
                {'message_id': 5, 'text': 'Poll question', 'poll': {'question': 'Test?'}},
            ],
        },
    },
]

print("Edge Case Test Results")
print("=" * 70)

for tc in test_cases:
    print(f"\n--- {tc['name']} ---")
    try:
        result = tc['adapter'].normalize(tc['data'])
        print(f"  Status:   OK")
        print(f"  Handle:   {result.profile.handle}")
        print(f"  Platform: {result.profile.platform}")
        print(f"  Posts:    {len(result.posts)}")
        
        if result.posts:
            media_types = [p.media_type for p in result.posts]
            print(f"  Media:    {media_types}")
            fwd = [p.is_amplification for p in result.posts]
            print(f"  Forwards: {fwd}")
    except Exception as e:
        print(f"  Status:   ERROR - {type(e).__name__}: {e}")

## 5. Behavioral Indicator Extraction

Demonstrate how normalized data feeds into behavioral indicator computation.
These indicators are used by the quality scorer and system prompt.

In [ ]:
# === Cell 8: Behavioral Indicator Extraction ===
# Compute behavioral indicators from normalized data

def extract_indicators(briefing_input: BriefingInput) -> dict:
    """Extract behavioral indicators from normalized BriefingInput."""
    indicators = {}
    posts = briefing_input.posts
    profile = briefing_input.profile
    
    if not posts:
        return {'error': 'No posts to analyze'}
    
    # 1. Posting frequency
    if len(posts) >= 2:
        timestamps = sorted(p.timestamp for p in posts if p.timestamp != datetime.min)
        if len(timestamps) >= 2:
            time_span = (timestamps[-1] - timestamps[0]).total_seconds() / 86400  # days
            if time_span > 0:
                indicators['posting_frequency'] = {
                    'posts_per_day': round(len(posts) / time_span, 1),
                    'time_span_days': round(time_span, 1),
                }
    
    # 2. Amplification ratio
    amplified = sum(1 for p in posts if p.is_amplification)
    original = len(posts) - amplified
    indicators['content_amplification'] = {
        'original_pct': round(original / len(posts) * 100, 1),
        'amplified_pct': round(amplified / len(posts) * 100, 1),
        'role_assessment': 'producer' if original > amplified else 'amplifier',
    }
    
    # 3. Amplification sources
    sources = {}
    for p in posts:
        if p.is_amplification and p.amplification_source:
            sources[p.amplification_source] = sources.get(p.amplification_source, 0) + 1
    indicators['amplification_sources'] = dict(sorted(sources.items(), key=lambda x: -x[1]))
    
    # 4. Media type distribution
    media_types = {}
    for p in posts:
        media_types[p.media_type] = media_types.get(p.media_type, 0) + 1
    indicators['media_distribution'] = media_types
    
    # 5. Engagement analysis
    if profile.platform == 'twitter':
        total_likes = sum(p.engagement.get('likes', 0) for p in posts)
        total_rts = sum(p.engagement.get('retweets', 0) for p in posts)
        indicators['engagement'] = {
            'avg_likes': round(total_likes / len(posts), 1),
            'avg_retweets': round(total_rts / len(posts), 1),
        }
    elif profile.platform == 'telegram':
        total_views = sum(p.engagement.get('views', 0) for p in posts)
        total_fwd = sum(p.engagement.get('forwards', 0) for p in posts)
        indicators['engagement'] = {
            'avg_views': round(total_views / len(posts), 1),
            'avg_forwards': round(total_fwd / len(posts), 1),
            'view_to_subscriber_ratio': round(total_views / len(posts) / max(profile.followers, 1) * 100, 1),
        }
    
    return indicators


# Extract indicators for both platforms
print("Twitter Behavioral Indicators")
print("=" * 60)
tw_indicators = extract_indicators(twitter_input)
print(json.dumps(tw_indicators, indent=2))

print(f"\n\nTelegram Behavioral Indicators")
print("=" * 60)
tg_indicators = extract_indicators(telegram_input)
print(json.dumps(tg_indicators, indent=2))

## 6. Conclusions

### Key Results
1. **TwitterAdapter** correctly normalizes API v2 data including retweet detection, entity extraction, and engagement metrics
2. **TelegramAdapter** handles MTProto data with forwarding metadata, view counts, and multiple media types
3. **Common format** (BriefingInput) allows platform-agnostic analysis downstream
4. **Edge cases** are handled gracefully — missing fields default to sensible values
5. **Behavioral indicators** can be extracted uniformly from normalized data

### Key Design Decisions
- `is_amplification` unifies Twitter retweets and Telegram forwards
- `engagement` dict preserves platform-specific metrics (likes vs views)
- `platform_metadata` stores fields unique to each platform
- `raw` field preserves original data for debugging

### Production Readiness
- Adapters are ready for integration with live API data
- Error handling for malformed API responses should be added (Sprint 4)
- Additional adapters (Facebook, YouTube) follow the same pattern